# 1 - YOLO Extraction

This notebook performs: (1) reading Excel metadata, (2) downloading PDFs, (3) running YOLOv10 inference and (4) saving raw, unresized PNG crops to Dataset/1_Raw_Crops/.

Notes:
- Crops are saved as lossless PNG (no resizing/padding here).
- Configure `PROJECT_ROOT` below if needed.

In [ ]:
# Imports (only what extraction needs)
from pathlib import Path
import platform
import os
import json
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
import cv2
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pymupdf as fitz
import torch
from openpyxl import load_workbook
# YOLOv10 detector used by this pipeline (DocLayout-YOLO wrapper)
from doclayout_yolo import YOLOv10

warnings.filterwarnings("ignore")
print('Imports ready')

In [ ]:
# Configuration dictionaries (kept similar to original notebook)
MODEL_CONFIG = {
    'name': 'doclayout_yolo_docstructbench_imgsz1024.pt',
    'confidence_threshold': 0.35,
    'fallback_confidence_threshold': 0.18,
    'target_classes': ['figure', 'picture', 'image'],
    'min_crop_size': 24,
}

PDF_CONFIG = {
    'zoom_extraction': 2.0,
    'batch_size': 4,
    'timeout': 60,
    'user_agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
}

EXCEL_CONFIG = {
    'sheet_index': 0,
    'id_column': 'Record Number',
    'link_column': 'PDF Link',
}

# PROJECT_ROOT (override if needed)
OS_TYPE = platform.system()
if OS_TYPE == 'Linux':
    PROJECT_ROOT = Path('/home/vasco/Tese_Vasco_Lnx/Drive_files_to_syncronize')
elif OS_TYPE == 'Windows':
    PROJECT_ROOT = Path(r'C:/path/to/project')
else:
    PROJECT_ROOT = Path.home() / 'Tese_Vasco_Lnx'

# Output folder for raw crops (exact folder name required)
PATHS = {
    'raw_crops': PROJECT_ROOT / '3 - Images Processing & DataSets' / '2-Images_DataSets'
        .3/ 'Test for last Query DataSet Extraction',
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

print('Configured PROJECT_ROOT:', PROJECT_ROOT)
print('Raw crops folder:', PATHS['raw_crops'])

In [ ]:
# Utility functions for extraction (network, excel, model, rendering, inference)
import os


In [ ]:
# Save raw crop as lossless PNG (minimal compression)
def save_raw_crop_png(crop_bgr, filepath):
    # Ensure parent exists
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)
    # cv2.imwrite expects BGR; save PNG with compression level 0 for speed
    try:
        cv2.imwrite(str(filepath), crop_bgr, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        return filepath
    except Exception as e:
        raise

In [ ]:
# Extraction pipeline: downloads PDFs, runs YOLO and saves raw PNG crops
def extract_images_from_patents(model, patents_df, id_column, link_column, output_dir, session, skip_existing=True):
    stats = {'total_patents': len(patents_df), 'patents_processed': 0, 'patents_skipped_existing': 0, 'patents_failed': 0, 'total_images': 0, 'details': []}
    device_mode = 0 if torch.cuda.is_available() else 'cpu'
    primary_conf = float(MODEL_CONFIG.get('confidence_threshold', 0.35))
    fallback_conf = float(MODEL_CONFIG.get('fallback_confidence_threshold', max(0.15, primary_conf * 0.6)))
    for patent_idx, (_, patent_row) in enumerate(patents_df.iterrows(), start=1):
        patent_id = normalize_patent_id(patent_row.get(id_column, ''))
        pdf_url = patent_row.get(link_column, '')
        print(f'[{patent_idx}/{stats[
]}] {patent_id}')
        if isinstance(pdf_url, str):
            pdf_url = pdf_url.strip()
        if not pdf_url or not (isinstance(pdf_url, str) and (pdf_url.startswith('http') or pdf_url.startswith('ftp'))):
            stats['patents_failed'] += 1
            stats['details'].append({'patent_id': patent_id, 'status': 'INVALID_URL', 'images': 0})
            continue
        patent_output_dir = Path(output_dir) / patent_id
        patent_output_dir.mkdir(parents=True, exist_ok=True)
        existing_pngs = list(patent_output_dir.glob('*.png'))
        if skip_existing and existing_pngs:
            stats['patents_skipped_existing'] += 1
            stats['details'].append({'patent_id': patent_id, 'status': 'SKIPPED_ALREADY_EXTRACTED', 'images': len(existing_pngs)})
            continue
        for old in existing_pngs:
            old.unlink(missing_ok=True)
        doc = None
        try:
            pdf_bytes = None
            for attempt in range(3):
                pdf_bytes = download_pdf_bytes(pdf_url, session=session, timeout=PDF_CONFIG['timeout'])
                if pdf_bytes is not None:
                    break
            if pdf_bytes is None:
                raise RuntimeError('Download failed')
            doc = fitz.open(stream=pdf_bytes, filetype='pdf')
            num_pages = len(doc)
            patent_images_count = 0
            batch_size = int(PDF_CONFIG['batch_size'])
            zoom = float(PDF_CONFIG['zoom_extraction'])
            for batch_start in range(0, num_pages, batch_size):
                batch_end = min(batch_start + batch_size, num_pages)
                batch_images = []
                batch_page_numbers = []
                for page_num in range(batch_start, batch_end):
                    try:
                        page = doc.load_page(page_num)
                        img_cv = render_pdf_page_to_bgr(page, zoom)
                        if img_cv is None or img_cv.size == 0:
                            continue
                        batch_images.append(img_cv)
                        batch_page_numbers.append(page_num + 1)
                    except Exception:
                        continue
                if not batch_images:
                    continue
                yolo_results, used_fallback = run_yolo_inference_with_fallback(model, batch_images, confidence=primary_conf, device=device_mode)
                for image_index, yolo_res in enumerate(yolo_results):
                    img_base = batch_images[image_index]
                    page_current = batch_page_numbers[image_index]
                    img_h, img_w = img_base.shape[:2]
                    target_boxes = extract_target_boxes(yolo_res, MODEL_CONFIG['target_classes'])
                    if len(target_boxes) == 0 and fallback_conf < primary_conf:
                        single_low, _ = run_yolo_inference_with_fallback(model, [img_base], confidence=fallback_conf, device=device_mode)
                        if len(single_low) > 0:
                            target_boxes = extract_target_boxes(single_low[0], MODEL_CONFIG['target_classes'])
                    for obj_idx, (class_name, x1, y1, x2, y2, _) in enumerate(target_boxes):
                        bbox = sanitize_bbox(x1, y1, x2, y2, img_width=img_w, img_height=img_h, min_size=int(MODEL_CONFIG.get('min_crop_size', 24)), pad=2)
                        if bbox is None:
                            continue
                        bx1, by1, bx2, by2 = bbox
                        crop = img_base[by1:by2, bx1:bx2]
                        if crop is None or crop.size == 0:
                            continue
                        filename = f"{patent_id}_p{page_current:03d}_o{obj_idx:02d}_{class_name}.png"
                        filepath = patent_output_dir / filename
                        try:
                            save_raw_crop_png(crop, filepath)
                            patent_images_count += 1
                        except Exception as e:
                            print('Save failed:', e)
            stats['patents_processed'] += 1
            stats['total_images'] += patent_images_count
            status = 'OK' if patent_images_count > 0 else 'NO_TARGET_DETECTIONS'
            stats['details'].append({'patent_id': patent_id, 'status': status, 'images': patent_images_count})
        except Exception as e:
            stats['patents_failed'] += 1
            stats['details'].append({'patent_id': patent_id, 'status': 'FAILED', 'images': 0, 'error': str(e)})
            print('Patent failed:', e)
        finally:
            if doc is not None:
                try:
                    doc.close()
                except Exception:
                    pass
    details_df = pd.DataFrame.from_records(stats['details'])
    return stats, details_df


In [ ]:
# Example usage: configure excel filename and run extraction (uncomment to run)
# NOTE: running this will download files and may take long; run interactively.
# model = load_model_with_fallback()
# session = create_http_session_with_retry()
# excel_filename = '521_P_very_restricted_dataset_06_04_26.xlsx'
# df_patents, excel_path = (pd.read_excel(find_file(excel_filename)), find_file(excel_filename))
# stats, details_df = extract_images_from_patents(model, df_patents, EXCEL_CONFIG['id_column'], EXCEL_CONFIG['link_column'], PATHS['raw_crops'], session, skip_existing=True)
# print(stats)
